# 01 — Data Audit & Ingestion

Build the competition-season-match inventory, download raw data, and parse into canonical tables.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import yaml, pandas as pd
from src.data.ingest import get_competitions, get_matches
from src.data.inventory import build_inventory, get_360_match_ids
from src.data.parse_events import parse_events
from src.data.parse_360 import parse_360_frames, get_frame_summary
from src.data.parse_lineups import parse_lineups
from src.data.join_pass_frames import build_pass_instances

with open("../configs/data.yaml") as f:
    cfg = yaml.safe_load(f)


## 1. Competition inventory

In [2]:
competitions = get_competitions()
print(f"Total competitions: {len(competitions)}")
competitions.head(10)


Total competitions: 75


,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-07-06T04:26:07.636270,2025-07-06T04:26:07.636270,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,None,None,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2024-09-28T01:57:35.846538,None,None,2024-09-28T01:57:35.846538
3,16,4,Europe,Champions League,male,False,False,2018/2019,2025-05-08T15:10:50.835274,2021-06-13T16:17:31.694,None,2025-05-08T15:10:50.835274
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,None,2024-02-13T02:35:28.134882
5,16,2,Europe,Champions League,male,False,False,2016/2017,2024-02-13T02:37:32.205154,2021-06-13T16:17:31.694,None,2024-02-13T02:37:32.205154
6,16,27,Europe,Champions League,male,False,False,2015/2016,2024-06-12T07:45:38.786894,2021-06-13T16:17:31.694,None,2024-06-12T07:45:38.786894
7,16,26,Europe,Champions League,male,False,False,2014/2015,2024-02-12T12:49:54.914228,2021-06-13T16:17:31.694,None,2024-02-12T12:49:54.914228
8,16,25,Europe,Champions League,male,False,False,2013/2014,2024-02-12T12:48:48.479157,2021-06-13T16:17:31.694,None,2024-02-12T12:48:48.479157
9,16,24,Europe,Champions League,male,False,False,2012/2013,2024-02-12T12:47:34.340413,2021-06-13T16:17:31.694,None,2024-02-12T12:47:34.340413


## 2. Match inventory with 360 coverage

In [3]:
inventory = build_inventory(cfg["statsbomb"]["competitions"])
print(inventory.groupby(["competition_name", "has_360"])["match_id"].count().unstack(fill_value=0))
inventory.head()


has_360           False  True 
competition_name              
FIFA World Cup       83     64
La Liga             833     35
Premier League      418      0


,competition_id,competition_name,season_id,season_name,match_id,has_events,has_lineups,has_360
0,2,Premier League,27,2015/2016,3753972,True,True,False
1,2,Premier League,27,2015/2016,3753973,True,True,False
2,2,Premier League,27,2015/2016,3753974,True,True,False
3,2,Premier League,27,2015/2016,3753975,True,True,False
4,2,Premier League,27,2015/2016,3753976,True,True,False


## 3. Parse a sample match

In [5]:

# Pick a match that actually has cached 360 frame data
import pathlib
raw_dir = pathlib.Path("../data/raw")
frame_files = sorted(raw_dir.glob("frames_*.parquet"))
if frame_files:
    sample_match_id = int(frame_files[0].stem.split("_")[1])
else:
    # Fallback: first match marked has_360 in inventory
    sample_match_id = int(inventory.loc[inventory["has_360"], "match_id"].iloc[0])
print(f"Sample match with 360: {sample_match_id}")

from src.data.ingest import get_events, get_360_frames
raw_events = get_events(sample_match_id)
raw_frames = get_360_frames(sample_match_id)

events = parse_events(raw_events)
frames = parse_360_frames(raw_frames)
frame_summary = get_frame_summary(frames)

print(f"Events: {len(events)}")
print(f"Frame rows: {len(frames)}")
print(f"Events with 360: {len(frame_summary)}")


Sample match with 360: 3857254
Events: 3680
Frame rows: 37653
Events with 360: 3157


## 4. Build pass_instances canonical table

In [6]:
pass_instances = build_pass_instances(events, frame_summary)
print(f"Pass instances: {len(pass_instances)}")
print(f"With 360: {pass_instances['has_360'].sum()}")
print(f"\nColumns: {list(pass_instances.columns)}")
pass_instances.head()


Pass instances: 557
With 360: 473

Columns: ['match_id', 'competition_id', 'season_id', 'event_uuid', 'possession_id', 'team_name', 'player_name', 'pass_recipient_name', 'minute', 'second', 'period', 'start_x', 'start_y', 'end_x', 'end_y', 'pass_length', 'pass_angle', 'pass_body_part', 'pass_height', 'pass_type', 'pass_outcome_name', 'under_pressure', 'pass_switch', 'pass_cross', 'has_360', 'n_visible_players', 'n_visible_teammates', 'n_visible_opponents', 'line_break', 'strict_line_break', 'loose_line_break', 'dangerous_progression_k', 'final_third_entry_k', 'box_entry_k', 'shot_within_k', 'threat_gain']


,match_id,competition_id,season_id,event_uuid,possession_id,team_name,player_name,pass_recipient_name,minute,second,...,n_visible_teammates,n_visible_opponents,line_break,strict_line_break,loose_line_break,dangerous_progression_k,final_third_entry_k,box_entry_k,shot_within_k,threat_gain
0,3857254,<NA>,<NA>,95ee43ac-863e-481f-81d5-8df75f8526ad,2,Denmark,Christian Dannemann Eriksen,Andreas Skov Olsen,0,1,...,9,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3857254,<NA>,<NA>,57ca09cc-0dfc-4789-a68c-506ab7f1a050,2,Denmark,Andreas Skov Olsen,Rasmus Nissen Kristensen,0,3,...,6,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3857254,<NA>,<NA>,f53c3831-2601-4ccd-b0f5-5503e165abcd,2,Denmark,Rasmus Nissen Kristensen,Kasper Dolberg,0,5,...,5,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3857254,<NA>,<NA>,99476cd4-869e-4991-a373-f60c2f2e4661,2,Denmark,Joakim Mæhle,None,0,9,...,8,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3857254,<NA>,<NA>,4709b249-432b-424b-b2af-5b9c3986709c,6,Denmark,Rasmus Nissen Kristensen,Andreas Skov Olsen,2,6,...,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Schema checks

In [7]:
required_cols = ["event_uuid", "match_id", "start_x", "start_y", "end_x", "end_y",
                 "pass_length", "has_360", "n_visible_teammates", "n_visible_opponents"]
missing = [c for c in required_cols if c not in pass_instances.columns]
print("Missing required columns:", missing or "None — all present ✓")
print("\nMissingness report:")
print(pass_instances.isnull().mean().sort_values(ascending=False).head(15))


Missing required columns: None — all present ✓

Missingness report:
competition_id             1.000000
season_id                  1.000000
strict_line_break          1.000000
line_break                 1.000000
final_third_entry_k        1.000000
box_entry_k                1.000000
loose_line_break           1.000000
dangerous_progression_k    1.000000
shot_within_k              1.000000
threat_gain                1.000000
pass_type                  0.899461
pass_outcome_name          0.842011
n_visible_teammates        0.150808
n_visible_opponents        0.150808
n_visible_players          0.150808
dtype: float64
